# Ensemble — BERT + RoBERTa
Combines your best RoBERTa predictions with your groupmate's best BERT predictions via **soft voting** (averaging predicted class probabilities), then computes the final Accuracy/Precision/Recall/F1 for the ENSEMBLE row.

**Before running:** upload these 4 files (from the two export notebooks) into this notebook's working directory:
- `bert_test_probs.npy`, `bert_test_labels.npy`
- `roberta_test_probs.npy`, `roberta_test_labels.npy`


In [2]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/tousiftasin/ensemble-npy-files/roberta_test_probs.npy
/kaggle/input/datasets/tousiftasin/ensemble-npy-files/bert_test_labels.npy
/kaggle/input/datasets/tousiftasin/ensemble-npy-files/bert_test_probs.npy
/kaggle/input/datasets/tousiftasin/ensemble-npy-files/roberta_test_labels.npy


In [3]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

bert_probs = np.load("/kaggle/input/datasets/tousiftasin/ensemble-npy-files/bert_test_probs.npy")
bert_labels = np.load("/kaggle/input/datasets/tousiftasin/ensemble-npy-files/bert_test_labels.npy")

roberta_probs = np.load("/kaggle/input/datasets/tousiftasin/ensemble-npy-files/roberta_test_probs.npy")
roberta_labels = np.load("/kaggle/input/datasets/tousiftasin/ensemble-npy-files/roberta_test_labels.npy")

print("BERT probs shape:", bert_probs.shape)
print("RoBERTa probs shape:", roberta_probs.shape)

assert np.array_equal(bert_labels, roberta_labels), (
    "Label arrays don't match! Check that both notebooks used the same "
    "random_state=42 split on the same dataset — the test sets must be identical and in the same order."
)

print("Label arrays match — safe to ensemble.")

BERT probs shape: (7033, 4)
RoBERTa probs shape: (7033, 4)
Label arrays match — safe to ensemble.


In [4]:
# Soft voting: average the two models' predicted probability distributions
ensemble_probs = (bert_probs + roberta_probs) / 2

ensemble_preds = np.argmax(ensemble_probs, axis=-1)
true_labels = bert_labels

ensemble_accuracy = accuracy_score(true_labels, ensemble_preds)
ensemble_precision = precision_score(true_labels, ensemble_preds, average="weighted")
ensemble_recall = recall_score(true_labels, ensemble_preds, average="weighted")
ensemble_f1 = f1_score(true_labels, ensemble_preds, average="weighted")

print("ENSEMBLE Results (BERT + RoBERTa, soft voting)")
print("Accuracy:", ensemble_accuracy)
print("Precision:", ensemble_precision)
print("Recall:", ensemble_recall)
print("F1 Score:", ensemble_f1)


ENSEMBLE Results (BERT + RoBERTa, soft voting)
Accuracy: 0.9623204891227073
Precision: 0.962325329320265
Recall: 0.9623204891227073
F1 Score: 0.9623220187308874


**For comparison**, individual model scores on the same test set:


In [5]:
bert_preds = np.argmax(bert_probs, axis=-1)
roberta_preds = np.argmax(roberta_probs, axis=-1)

print("BERT alone      — Acc: {:.4f}, F1: {:.4f}".format(
    accuracy_score(true_labels, bert_preds),
    f1_score(true_labels, bert_preds, average="weighted")
))

print("RoBERTa alone   — Acc: {:.4f}, F1: {:.4f}".format(
    accuracy_score(true_labels, roberta_preds),
    f1_score(true_labels, roberta_preds, average="weighted")
))

print("Ensemble        — Acc: {:.4f}, F1: {:.4f}".format(
    ensemble_accuracy,
    ensemble_f1
))


BERT alone      — Acc: 0.9571, F1: 0.9571
RoBERTa alone   — Acc: 0.9619, F1: 0.9619
Ensemble        — Acc: 0.9623, F1: 0.9623
